# Build 01 — SFP Loop Simulation

> *Synthetic Self-Fulfilling Prophecy Loop in Insurance Fraud Detection*

**EN:** Simulates a 5-generation feedback loop. Initial model trained on a random seed → investigates top-k% → only investigated claims get labels → next model trained on those biased labels → repeat. The simulation gives us **ground truth** `true_fraud` (impossible in production) so we can measure how badly observed labels diverge from reality.

**KR:** 5세대 피드백 루프를 시뮬레이션. 랜덤 시드로 학습한 초기 모델 → 상위 k% 조사 → 조사된 클레임만 라벨 부여 → 그 편향된 라벨로 다음 모델 학습 → 반복. 시뮬레이션이라 **ground truth** `true_fraud`를 알 수 있고 (프로덕션에서는 불가능), 관측 라벨이 실제와 얼마나 괴리되는지 측정 가능.

## 1. Imports

**EN:** scikit-learn for logistic regression and metrics; `np.random.default_rng` for reproducible randomness.

**KR:** scikit-learn으로 로지스틱 회귀와 메트릭; 재현 가능한 난수는 `np.random.default_rng`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 2. Synthetic Claims Generator

**EN:** Generates `n_claims` synthetic claims with features (amount, hour, postcode risk, prior claims, type) and a latent true fraud probability following: `logit(P(fraud)) = -2.5 + 0.8·high_amount + 0.6·night + 1.2·high_postcode + 0.4·prior_claims`.

**KR:** `n_claims` 합성 클레임 생성. feature(금액·시간·우편번호 위험도·이전 클레임·유형)와 잠재 fraud 확률은 다음 공식: `logit(P(fraud)) = -2.5 + 0.8·high_amount + 0.6·night + 1.2·high_postcode + 0.4·prior_claims`.

In [ ]:
def generate_claims(n_claims: int = 50_000, seed: int = 42) -> pd.DataFrame:
    """
    Generate synthetic insurance claims with TRUE fraud labels.

    Features:
        - claim_amount: log-normal (fraud claims tend to be larger)
        - claim_hour: time of day (night claims higher fraud)
        - postcode_risk: postcode-level risk (0=low, 1=high)
        - prior_claims: number of prior claims by claimant
        - claim_type: categorical (theft, damage, injury)

    True fraud model (latent):
        logit(P(fraud)) = -2.5 + 0.8*high_amount + 0.6*night +
                          1.2*high_postcode_risk + 0.4*prior_claims
    """
    rng = np.random.default_rng(seed)

    n = n_claims

    # Claim features
    claim_amount_raw = rng.lognormal(mean=7.5, sigma=1.2, size=n)  # GBP
    claim_hour = rng.integers(0, 24, size=n)
    postcode_risk = rng.beta(1.5, 3.0, size=n)  # 0-1, right-skewed (most low risk)
    prior_claims = rng.poisson(0.5, size=n).clip(0, 5)
    claim_type = rng.choice(['theft', 'damage', 'injury'], p=[0.3, 0.5, 0.2], size=n)

    # Derived features
    high_amount = (claim_amount_raw > np.percentile(claim_amount_raw, 75)).astype(float)
    night_claim = ((claim_hour >= 22) | (claim_hour <= 5)).astype(float)
    high_postcode = (postcode_risk > 0.6).astype(float)

    # True fraud probability (the ground truth we know but the model doesn't)
    log_odds = (-2.5
                + 0.8 * high_amount
                + 0.6 * night_claim
                + 1.2 * high_postcode
                + 0.4 * prior_claims
                + rng.normal(0, 0.3, n))  # individual noise

    true_fraud_prob = 1 / (1 + np.exp(-log_odds))
    true_fraud = rng.binomial(1, true_fraud_prob).astype(int)

    df = pd.DataFrame({
        'claim_id': [f'CLM{i:06d}' for i in range(n)],
        'claim_amount': claim_amount_raw.round(2),
        'claim_hour': claim_hour,
        'postcode_risk': postcode_risk.round(4),
        'prior_claims': prior_claims,
        'claim_type': claim_type,
        'high_amount': high_amount,
        'night_claim': night_claim,
        'high_postcode': high_postcode,
        'true_fraud': true_fraud,          # GROUND TRUTH — not available in production
        'true_fraud_prob': true_fraud_prob.round(4),
    })

    print(f"Generated {n:,} claims | True fraud rate: {true_fraud.mean():.3f} ({true_fraud.sum():,} frauds)")
    return df

## 3. Model Training & Scoring

**EN:** Logistic regression trained on `observed_fraud` (the biased label — *not* `true_fraud`). `class_weight='balanced'` corrects for low fraud base rate.

**KR:** `observed_fraud`(편향된 라벨 — `true_fraud` 아님)로 로지스틱 회귀 학습. 낮은 fraud 기저율 보정용 `class_weight='balanced'`.

In [ ]:
FEATURE_COLS = ['high_amount', 'night_claim', 'high_postcode', 'prior_claims']


def train_model(training_df: pd.DataFrame, weights: np.ndarray = None) -> LogisticRegression:
    """Train logistic regression fraud model on OBSERVED (biased) labels."""
    X = training_df[FEATURE_COLS].values
    y = training_df['observed_fraud'].values  # NOT true_fraud — this is the biased label
    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X, y, sample_weight=weights)
    return model


def score_claims(model: LogisticRegression, df: pd.DataFrame) -> np.ndarray:
    """Return fraud risk scores (probability of fraud) for all claims."""
    X = df[FEATURE_COLS].values
    return model.predict_proba(X)[:, 1]

## 4. Investigation Policy

**EN:** Top-k% by score get investigated. The `epsilon` knob adds random exploration. `observed_fraud` is NaN for uninvestigated claims — that's what makes labels selective.

**KR:** 점수 상위 k%만 조사. `epsilon` 노브로 무작위 탐색 추가. 조사 안 한 클레임의 `observed_fraud`는 NaN — 이게 라벨을 selective하게 만드는 메커니즘.

In [ ]:
def apply_investigation_policy(
    df: pd.DataFrame,
    model_scores: np.ndarray,
    investigation_rate: float = 0.25,
    random_exploration_rate: float = 0.0,
    rng: np.random.Generator = None
) -> pd.DataFrame:
    """
    Assign investigation decisions based on model scores.

    Args:
        investigation_rate: Fraction of claims to investigate (model-driven)
        random_exploration_rate: Additional random sample (epsilon in epsilon-greedy)

    Returns:
        DataFrame with 'investigated' and 'observed_fraud' columns added.
    """
    if rng is None:
        rng = np.random.default_rng(0)

    n = len(df)
    threshold = np.percentile(model_scores, 100 * (1 - investigation_rate))

    # Model-driven investigations
    model_investigate = (model_scores >= threshold)

    # Random exploration (epsilon-greedy)
    random_investigate = rng.random(n) < random_exploration_rate

    investigated = (model_investigate | random_investigate)

    # Fraud is ONLY observable in investigated claims
    # Add small investigator error (2% false negative rate)
    investigator_fnr = 0.02
    observed_fraud = np.where(
        investigated,
        np.where(df['true_fraud'].values == 1,
                 (rng.random(n) > investigator_fnr).astype(int),  # true fraud detected with 98% prob
                 0),
        np.nan  # fraud label UNKNOWN for uninvestigated claims
    )

    result = df.copy()
    result['model_score'] = model_scores.round(4)
    result['investigated'] = investigated.astype(int)
    result['observed_fraud'] = observed_fraud

    return result

## 5. Full Multi-Generation Simulation

**EN:** Trains 5 sequential models, each using the previous version's biased labels. Tracks per-version: AUC on investigated (biased view — looks great), TRUE recall (vs ground truth — drops), precision, loop amplification, blind-spot fraction.

**KR:** 5개의 순차 모델을 학습, 각각 이전 버전의 편향된 라벨 사용. 버전별 추적: 조사된 AUC(편향 시각 — 좋아 보임), TRUE recall(ground truth 대비 — 하락), precision, 루프 증폭, blind spot 비율.

In [ ]:
def run_full_simulation(
    n_claims: int = 50_000,
    n_versions: int = 5,
    investigation_rate: float = 0.25,
    epsilon: float = 0.0,  # random exploration rate
    seed: int = 42
) -> tuple[pd.DataFrame, list[dict]]:
    """
    Simulate the full SFP loop across multiple model versions.

    Returns:
        full_df: All claims with columns for each model version's scores
        metrics: List of per-version performance metrics
    """
    rng = np.random.default_rng(seed)

    # Generate ground truth
    df = generate_claims(n_claims=n_claims, seed=seed)

    # Seed model: train on a small RANDOM sample (unbiased initial seed)
    seed_size = int(n_claims * 0.05)  # 5% random seed
    seed_idx = rng.choice(len(df), size=seed_size, replace=False)
    seed_df = df.iloc[seed_idx].copy()
    seed_df['observed_fraud'] = seed_df['true_fraud']  # unbiased seed

    current_model = train_model(seed_df)

    metrics_per_version = []
    all_training_dfs = []

    for version in range(1, n_versions + 1):
        # Score all claims with current model
        scores = score_claims(current_model, df)

        # Apply investigation policy
        versioned_df = apply_investigation_policy(
            df, scores,
            investigation_rate=investigation_rate,
            random_exploration_rate=epsilon,
            rng=rng
        )

        # Store version scores
        df[f'model_v{version}_score'] = scores
        df[f'model_v{version}_investigated'] = versioned_df['investigated']
        df[f'model_v{version}_observed_fraud'] = versioned_df['observed_fraud']

        # ── Performance metrics (against TRUE fraud labels — ground truth) ──
        investigated_mask = versioned_df['investigated'] == 1
        investigated_df = versioned_df[investigated_mask].copy()

        # AUC on investigated claims only (biased view)
        auc_biased = roc_auc_score(
            investigated_df['true_fraud'],
            investigated_df['model_score']
        )

        # True recall: what fraction of ALL true fraud did we find?
        true_fraud_found = (
            investigated_df['true_fraud'] == 1
        ).sum()
        true_recall = true_fraud_found / df['true_fraud'].sum()

        # Precision on investigated claims
        precision = precision_score(
            investigated_df['true_fraud'],
            (investigated_df['model_score'] > 0.5).astype(int),
            zero_division=0
        )

        # Loop amplification: fraud rate in top quartile vs true population rate
        top_quartile_mask = scores >= np.percentile(scores, 75)
        fraud_rate_top = df.loc[top_quartile_mask, 'true_fraud'].mean()
        true_fraud_rate = df['true_fraud'].mean()
        loop_amplification = fraud_rate_top / true_fraud_rate if true_fraud_rate > 0 else 1.0

        # Blind spot: fraction of true fraud in NEVER-investigated segments
        # Segment by high_postcode x night_claim
        never_investigated_mask = versioned_df['investigated'] == 0
        fraud_in_blind_spot = (
            df.loc[never_investigated_mask, 'true_fraud'] == 1
        ).sum()
        blind_spot_fraction = fraud_in_blind_spot / df['true_fraud'].sum()

        metrics = {
            'version': version,
            'n_investigated': investigated_mask.sum(),
            'investigation_rate': investigated_mask.mean(),
            'observed_fraud_rate': investigated_df['observed_fraud'].mean(),
            'auc_on_investigated': round(auc_biased, 4),
            'true_recall': round(true_recall, 4),
            'precision': round(precision, 4),
            'loop_amplification_factor': round(loop_amplification, 3),
            'blind_spot_fraud_fraction': round(blind_spot_fraction, 4),
        }
        metrics_per_version.append(metrics)

        print(f"\n── Model v{version} ──────────────────────────────")
        print(f"  Investigated:       {metrics['n_investigated']:,} ({metrics['investigation_rate']:.1%})")
        print(f"  Observed fraud rate:{metrics['observed_fraud_rate']:.3f}")
        print(f"  AUC (biased):       {metrics['auc_on_investigated']:.4f}")
        print(f"  TRUE Recall:        {metrics['true_recall']:.4f}   ← real coverage")
        print(f"  Precision:          {metrics['precision']:.4f}")
        print(f"  Loop Amplification: {metrics['loop_amplification_factor']:.2f}x")
        print(f"  Blind Spot Fraud:   {metrics['blind_spot_fraud_fraction']:.1%} of all fraud undetectable")

        # Train next version model on OBSERVED (biased) labels
        training_df = versioned_df[versioned_df['investigated'] == 1].copy()
        training_df = training_df.dropna(subset=['observed_fraud'])
        training_df['observed_fraud'] = training_df['observed_fraud'].astype(int)
        all_training_dfs.append(training_df)

        # Combine all historical training data
        combined_training = pd.concat(all_training_dfs, ignore_index=True)
        current_model = train_model(combined_training)

    return df, metrics_per_version

## 6. Visualisation — Loop Degradation

**EN:** Four panels: AUC (biased — rising), TRUE recall (real — falling), blind-spot fraction (rising), amplification factor (rising). Standard metrics improve while reality worsens.

**KR:** 4패널: AUC(편향 — 상승), TRUE recall(실제 — 하락), blind spot 비율(상승), 증폭 계수(상승). 표준 지표는 좋아지고 실제 성능은 나빠지는 분리 현상.

In [ ]:
def plot_loop_degradation(metrics_list: list[dict], title_suffix: str = "") -> None:
    """Plot how metrics evolve across model versions."""
    versions = [m['version'] for m in metrics_list]
    auc_scores = [m['auc_on_investigated'] for m in metrics_list]
    true_recalls = [m['true_recall'] for m in metrics_list]
    blind_spots = [m['blind_spot_fraud_fraction'] for m in metrics_list]
    amplifications = [m['loop_amplification_factor'] for m in metrics_list]

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(f'SFP Loop Effect Across Model Versions {title_suffix}', fontsize=14)

    axes[0, 0].plot(versions, auc_scores, 'o-', color='steelblue')
    axes[0, 0].set_title('AUC (on investigated claims — BIASED)')
    axes[0, 0].set_xlabel('Model Version')
    axes[0, 0].set_ylabel('AUC-ROC')
    axes[0, 0].set_ylim(0.5, 1.0)

    axes[0, 1].plot(versions, true_recalls, 'o-', color='tomato')
    axes[0, 1].set_title('TRUE Recall (against ground truth)')
    axes[0, 1].set_xlabel('Model Version')
    axes[0, 1].set_ylabel('Recall')
    axes[0, 1].set_ylim(0, 1.0)

    axes[1, 0].plot(versions, blind_spots, 'o-', color='darkorange')
    axes[1, 0].set_title('Blind Spot Fraction (fraud never found)')
    axes[1, 0].set_xlabel('Model Version')
    axes[1, 0].set_ylabel('Fraction of True Fraud')
    axes[1, 0].set_ylim(0, 1.0)

    axes[1, 1].plot(versions, amplifications, 'o-', color='purple')
    axes[1, 1].axhline(1.0, color='grey', linestyle='--', label='No amplification')
    axes[1, 1].set_title('Loop Amplification Factor')
    axes[1, 1].set_xlabel('Model Version')
    axes[1, 1].set_ylabel('Amplification (×)')
    axes[1, 1].legend()

    plt.tight_layout()
    plt.savefig('sfp_loop_degradation.png', dpi=150)
    plt.show()
    print("Saved: sfp_loop_degradation.png")

## 7. Policy Comparison Function

**EN:** Runs the full simulation under multiple epsilon values. Returns a tidy DataFrame for plotting and statistical comparison.

**KR:** 여러 epsilon 값으로 전체 시뮬레이션 반복. plotting과 통계 비교용 tidy DataFrame 반환.

In [ ]:
def compare_policies(
    n_claims: int = 20_000,
    n_versions: int = 5,
    epsilons: list[float] = [0.0, 0.05, 0.10, 0.20]
) -> pd.DataFrame:
    """Compare SFP loop under different epsilon-greedy exploration rates."""
    results = []
    for eps in epsilons:
        print(f"\n{'='*50}")
        print(f"Policy: ε = {eps} ({'pure model' if eps == 0 else f'{eps:.0%} random exploration'})")
        print(f"{'='*50}")
        _, metrics = run_full_simulation(
            n_claims=n_claims, n_versions=n_versions,
            investigation_rate=0.25, epsilon=eps, seed=42
        )
        for m in metrics:
            m['epsilon'] = eps
            results.append(m)

    return pd.DataFrame(results)

---

# 👁️ Section A — Hands-on Demo: Run the Loop Yourself

> *Below we generate 10,000 synthetic claims, run 5 model generations, and print everything as it happens.*

**EN:** This is the live demonstration. Watch the metrics evolve across versions — you should see AUC rising, TRUE recall falling, blind spots growing. That divergence *is* the SFP loop.

**KR:** 라이브 시연. 버전마다 메트릭 변화를 관찰 — AUC는 상승, TRUE recall은 하락, blind spot은 증가해야 함. 이 괴리가 바로 SFP 루프.

## A.1 Step 1 — Generate ground-truth claims

**EN:** Generate 10k claims with known `true_fraud` labels. Print a sample so you see the feature columns and the ground truth.

**KR:** 10k 클레임 생성(ground truth `true_fraud` 라벨 포함). feature 칼럼과 ground truth를 sample로 출력.

In [ ]:
np.random.seed(42)
df_truth = generate_claims(n_claims=10_000, seed=42)

print("\n📋 Ground truth sample (first 5 rows):")
print(df_truth[['claim_id', 'claim_amount', 'claim_hour', 'postcode_risk',
                'prior_claims', 'true_fraud_prob', 'true_fraud']].head().to_string(index=False))

print(f"\n📊 Ground truth fraud rate: {df_truth['true_fraud'].mean():.3%}")
print(f"   Total true frauds: {df_truth['true_fraud'].sum():,} of {len(df_truth):,} claims")

## A.2 Step 2 — Run 5-generation loop (epsilon = 0, pure loop)

**EN:** Each generation: train → score → investigate top-25% → observe biased labels → retrain. Per-version metrics print as the loop runs.

**KR:** 각 세대: 학습 → 점수화 → 상위 25% 조사 → 편향된 라벨 관측 → 재학습. 루프가 돌면서 버전별 메트릭이 출력됨.

In [ ]:
print("\n" + "═" * 60)
print("  Running 5-generation SFP loop (ε = 0, no exploration)")
print("═" * 60)

df_loop, metrics_loop = run_full_simulation(
    n_claims=10_000,
    n_versions=5,
    investigation_rate=0.25,
    epsilon=0.0,
    seed=42,
)

## A.3 Step 3 — Summary table across versions

**EN:** All five versions in one table. Watch `auc_on_investigated` (biased — should rise) vs `true_recall` (real — should fall).

**KR:** 다섯 버전을 표 하나로. `auc_on_investigated`(편향 — 상승)과 `true_recall`(실제 — 하락)을 비교 관찰.

In [ ]:
summary = pd.DataFrame(metrics_loop)
print("\n📊 Per-version summary (ε = 0):")
print(summary[['version', 'investigation_rate', 'auc_on_investigated',
               'true_recall', 'loop_amplification_factor',
               'blind_spot_fraud_fraction']].to_string(index=False))

print(f"\n📉 Recall change: v1 → v5: "
      f"{summary['true_recall'].iloc[0]:.3f} → {summary['true_recall'].iloc[-1]:.3f} "
      f"({(summary['true_recall'].iloc[-1] - summary['true_recall'].iloc[0]):+.3f})")
print(f"📈 AUC change (biased):       "
      f"{summary['auc_on_investigated'].iloc[0]:.3f} → {summary['auc_on_investigated'].iloc[-1]:.3f}")
print()
print("👉 The model APPEARS to improve (AUC rises) while it actually loses recall (real metric falls).")
print("   This is the self-fulfilling prophecy in action.")

## A.4 Step 4 — Plot loop degradation

In [ ]:
plot_loop_degradation(metrics_loop, title_suffix='(ε=0, pure loop)')

## A.5 Step 5 — Compare ε=0 vs ε=0.05 exploration

**EN:** Run the same simulation with 5% random exploration. Compare final recall side-by-side. The improvement quantifies the value of randomisation strategies (Build 05).

**KR:** 동일 시뮬레이션을 5% 무작위 탐색으로 재실행. 최종 recall 비교. 개선폭이 randomisation 전략(Build 05)의 가치를 정량화.

In [ ]:
print("\n" + "═" * 60)
print("  Running 5-generation loop with ε = 0.05 exploration")
print("═" * 60)
df_eps, metrics_eps = run_full_simulation(
    n_claims=10_000, n_versions=5, investigation_rate=0.25, epsilon=0.05, seed=42
)

print("\n📊 Side-by-side comparison:")
print(f"{'Version':<10} {'TRUE recall (ε=0)':<22} {'TRUE recall (ε=0.05)':<24} {'Δ'}")
print("─" * 70)
for m0, m1 in zip(metrics_loop, metrics_eps):
    delta = m1['true_recall'] - m0['true_recall']
    print(f"  v{m0['version']:<7} {m0['true_recall']:<22.4f} {m1['true_recall']:<24.4f} {delta:+.4f}")
print()
print("👉 Even 5% random exploration recovers measurable recall in later versions.")
print("   Build 05 explores this trade-off systematically.")

## 🎯 What you should observe

**EN:**
- `auc_on_investigated` *rises* with version (the model gets better at scoring claims similar to ones it's already seen)
- `true_recall` *falls* (the model loses sight of fraud in segments it stops investigating)
- `loop_amplification_factor` *grows* (top quartile is increasingly fraud-concentrated)
- `blind_spot_fraud_fraction` *grows* (more fraud lives in never-investigated segments)
- With `ε=0.05`, the loss is partially recovered — that's the case for randomisation

**KR:**
- `auc_on_investigated`이 버전이 올라갈수록 **상승** (이미 본 비슷한 클레임 점수화에 능숙해짐)
- `true_recall`이 **하락** (조사를 중단한 segment의 fraud를 놓치기 시작)
- `loop_amplification_factor` **증가** (top quartile에 fraud 집중도 증가)
- `blind_spot_fraud_fraction` **증가** (영영 조사 안 되는 segment에 fraud가 쌓임)
- `ε=0.05`에서는 손실이 부분적으로 회복 — 이게 randomisation의 근거